In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))
import pandas as pd
from src.utils import load_data, clean_dates, standardize_states, fix_pincodes, process_and_save
from src.config import FILES

# Ensure pandas shows all rows when we print unique states
pd.set_option('display.max_rows', 100)

In [2]:
all_unique_states = set()

for key in FILES.keys():
    df = load_data(key)
    # Strip and lower just to see the raw variations safely
    raw_states = df['state'].astype(str).str.strip().str.lower().unique()
    all_unique_states.update(raw_states)

print(f"\nTotal unique state variations found: {len(all_unique_states)}")
print(sorted(list(all_unique_states)))

Loading aadhar_biometric_merged.csv...
Loading aadhar_enrolment_merged.csv...
Loading aadhar_demographic_merged.csv...

Total unique state variations found: 60
['100000', 'andaman & nicobar islands', 'andaman and nicobar islands', 'andhra pradesh', 'arunachal pradesh', 'assam', 'balanagar', 'bihar', 'chandigarh', 'chhatisgarh', 'chhattisgarh', 'dadra & nagar haveli', 'dadra and nagar haveli', 'dadra and nagar haveli and daman and diu', 'daman & diu', 'daman and diu', 'darbhanga', 'delhi', 'goa', 'gujarat', 'haryana', 'himachal pradesh', 'jaipur', 'jammu & kashmir', 'jammu and kashmir', 'jharkhand', 'karnataka', 'kerala', 'ladakh', 'lakshadweep', 'madanapalle', 'madhya pradesh', 'maharashtra', 'manipur', 'meghalaya', 'mizoram', 'nagaland', 'nagpur', 'odisha', 'orissa', 'pondicherry', 'puducherry', 'punjab', 'puttenahalli', 'raja annamalai puram', 'rajasthan', 'sikkim', 'tamil nadu', 'tamilnadu', 'telangana', 'the dadra and nagar haveli and daman and diu', 'tripura', 'uttar pradesh', 'ut

In [3]:
import pprint

# Assuming all_unique_states still holds your 60 unique values
sorted_states = sorted([str(s).strip().lower() for s in all_unique_states])

# Write the full dictionary to a text file
output_path = 'state_mapping_full.txt'
with open(output_path, 'w') as f:
    f.write("STATE_MAPPING = {\n")
    for state in sorted_states:
        f.write(f"    '{state}': 'correct_name_here',\n")
    f.write("}\n")

print(f"✅ Full mapping successfully saved to {output_path} without truncation!")

✅ Full mapping successfully saved to state_mapping_full.txt without truncation!


In [4]:
# The list of shifted/corrupted states we found
corrupted_states = [
    '100000', 'balanagar', 'darbhanga', 'jaipur', 
    'madanapalle', 'nagpur', 'puttenahalli', 'raja annamalai puram'
]

total_corrupted = 0
total_rows = 0

print("--- CORRUPTED ROWS IMPACT ANALYSIS ---\n")

for key in FILES.keys():
    df = load_data(key)
    
    # Temporarily lower/strip state names to match our list safely
    temp_state = df['state'].astype(str).str.strip().str.lower()
    
    # Count how many rows match our corrupted list
    mask = temp_state.isin(corrupted_states)
    count = mask.sum()
    
    total_corrupted += count
    total_rows += len(df)
    
    print(f"Dataset: {FILES[key]}")
    print(f"Affected Rows: {count} out of {len(df):,}")
    
    # If we found corrupted rows, let's print the first 2 to see WHAT shifted
    if count > 0:
        print("Example of the shifted data:")
        display(df[mask].head(2))
    print("-" * 50)

# Calculate final percentage
percentage_lost = (total_corrupted / total_rows) * 100
print(f"\nTotal Corrupted Rows Across All Datasets: {total_corrupted}")
print(f"Total Rows Overall: {total_rows:,}")
print(f"Percentage of Data to be Dropped: {percentage_lost:.5f}%")

--- CORRUPTED ROWS IMPACT ANALYSIS ---

Loading aadhar_biometric_merged.csv...
Dataset: aadhar_biometric_merged.csv
Affected Rows: 0 out of 1,861,108
--------------------------------------------------
Loading aadhar_enrolment_merged.csv...
Dataset: aadhar_enrolment_merged.csv
Affected Rows: 22 out of 1,006,029
Example of the shifted data:


,date,state,district,pincode,age_0_5,age_5_17,age_18_greater
23108,02-09-2025,100000,100000,100000,0,0,3
46946,03-09-2025,100000,100000,100000,0,0,1


--------------------------------------------------
Loading aadhar_demographic_merged.csv...
Dataset: aadhar_demographic_merged.csv
Affected Rows: 13 out of 2,071,700
Example of the shifted data:


,date,state,district,pincode,demo_age_5_17,demo_age_17_
222931,16-12-2025,Darbhanga,Near University Thana,846004,0,1
332384,16-12-2025,Darbhanga,Near University Thana,846004,0,1


--------------------------------------------------

Total Corrupted Rows Across All Datasets: 35
Total Rows Overall: 4,938,837
Percentage of Data to be Dropped: 0.00071%


In [5]:
# Process all three datasets
df_bio_clean = process_and_save('biometric')
df_demo_clean = process_and_save('demographic')
df_enrol_clean = process_and_save('enrolment')

# Sanity Check
print("--- SANITY CHECK ---")
print("Biometric Shape:", df_bio_clean.shape)
print("Demographic Shape:", df_demo_clean.shape)
print("Enrolment Shape:", df_enrol_clean.shape)

# Check if states consolidated down to 36
print("\nUnique states in Enrolment after cleaning:", df_enrol_clean['state'].nunique())
print("Data Types:\n", df_enrol_clean.dtypes)

Loading aadhar_biometric_merged.csv...
  -> Saved clean data to c:\Users\Asus\Documents\UIDAI Data Hackathon 2026\data\processed\cleaned_aadhar_biometric_merged.csv

Loading aadhar_demographic_merged.csv...
  -> Removed 13 corrupted rows.
  -> Saved clean data to c:\Users\Asus\Documents\UIDAI Data Hackathon 2026\data\processed\cleaned_aadhar_demographic_merged.csv

Loading aadhar_enrolment_merged.csv...
  -> Removed 22 corrupted rows.
  -> Saved clean data to c:\Users\Asus\Documents\UIDAI Data Hackathon 2026\data\processed\cleaned_aadhar_enrolment_merged.csv

--- SANITY CHECK ---
Biometric Shape: (1861108, 6)
Demographic Shape: (2071687, 6)
Enrolment Shape: (1006007, 7)

Unique states in Enrolment after cleaning: 36
Data Types:
 date              datetime64[ns]
state                     object
district                  object
pincode                   object
age_0_5                    int64
age_5_17                   int64
age_18_greater             int64
dtype: object


# 📊 Phase 0 Findings: Data Quality & Preprocessing

## Executive Summary
Real-world government datasets require rigorous standardization before analysis. During our Phase 0 pipeline, we encountered and resolved several critical data quality issues across the ~4.9 million rows of merged Aadhaar data.

### 1. State Name Fragmentation (Standardization)
* **The Issue:** The raw data contained **60 unique state designations** for India's 36 official States and Union Territories. This included typographical errors (e.g., `west bengli`, `karnataka `), capitalization inconsistencies, and outdated UT names (e.g., separate entries for Daman & Diu).
* **The Fix:** Implemented a scalable dictionary-mapping function in `utils.py` to consolidate all variations into the official 36 State/UT titles. This prevents map-charting errors in Tableau and ensures aggregations are accurate.

### 2. Shifted & Corrupted Rows (Integrity Check)
* **The Issue:** We identified structural corruption in the CSVs where data had shifted horizontally (e.g., city names like `jaipur` or pincodes like `100000` appearing in the `state` column). This shift corrupts the numeric biometric/demographic counts for those rows.
* **Impact Analysis:** Before blindly dropping data, we quantified the "blast radius."
  * **Corrupted Rows Found:** 35 
  * **Total Rows Evaluated:** 4,938,837
  * **Data Loss Percentage:** **0.00071%**
* **The Fix:** Because the corruption is statistically insignificant (< 0.001%), we safely dropped these 35 rows via a dynamic `'DROP'` tag in our configuration mapping. Attempting to artificially "shift" the data back would risk introducing synthetic errors for zero analytical gain.

### 3. Date & Pincode Formatting
* **Dates:** Successfully cast the string `DD-MM-YYYY` formats to datetime objects to enable Phase 2 time-series analysis. Confirmed bounds are contained within the 2025 calendar year.
* **Pincodes:** Standardized to 6-character strings, adding leading zeros where pandas integer conversion had aggressively truncated them.

**Conclusion:** The datasets residing in `data/processed/` are now pristine, consolidated, and cleared for Exploratory Data Analysis.